# 🌱 Plantica AI: State-of-the-Art ConvNeXt-V2 Plant Disease & Non-Plant Detection
### 🚀 Architecture: `ConvNeXt-Base (384x384)` with PyTorch, TIMM, and ONNX Runtime Export
---
This notebook trains a production-grade Vision model for plant disease diagnosis and non-plant object filtering across **387 classes**.

In [ ]:
# 1. Install & Verify Required SOTA CV Libraries
!pip install -q timm albumentations onnx onnxruntime scikit-learn seaborn matplotlib tqdm

In [ ]:
import os
import sys
import json
import time
import random
import shutil
import zipfile
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, top_k_accuracy_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Set deterministic seed
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⚡ Computation Device: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU Name: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Auto-Detect Dataset Path
# Automatically search for the dataset directory in Kaggle or Colab
DATASET_ROOT = None

search_locations = [
    '/kaggle/input',
    './dataset',
    '../dataset',
    '/content/dataset'
]

for loc in search_locations:
    if os.path.exists(loc):
        for root, dirs, files in os.walk(loc):
            # Look for directory with many class subfolders
            valid_subdirs = [d for d in dirs if not d.startswith('.') and not d.startswith('__')]
            if len(valid_subdirs) > 20:  # Found root dataset folder
                DATASET_ROOT = root
                break
        if DATASET_ROOT:
            break

if not DATASET_ROOT:
    # Fallback to direct input
    DATASET_ROOT = input("Enter dataset path: ").strip()

print(f"📂 Detected Dataset Root: {DATASET_ROOT}")

# Collect all classes
classes = sorted([d for d in os.listdir(DATASET_ROOT) if os.path.isdir(os.path.join(DATASET_ROOT, d)) and not d.startswith('.')])
class_to_idx = {cls_name: i for i, cls_name in enumerate(classes)}
idx_to_class = {i: cls_name for i, cls_name in enumerate(classes)}
num_classes = len(classes)

print(f"✅ Total Classes Found: {num_classes}")
print(f"Sample classes: {classes[:5]} ... {classes[-5:]}")

In [ ]:
# 3. Gather All Image Paths and Split (80% Train, 10% Val, 10% Test)
image_paths = []
image_labels = []
supported_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')

for cls_name in tqdm(classes, desc="Scanning classes"):
    cls_dir = os.path.join(DATASET_ROOT, cls_name)
    for fname in os.listdir(cls_dir):
        if fname.lower().endswith(supported_extensions):
            image_paths.append(os.path.join(cls_dir, fname))
            image_labels.append(class_to_idx[cls_name])

print(f"📸 Total Valid Images: {len(image_paths)}")

# Stratified Split
train_paths, val_test_paths, train_labels, val_test_labels = train_test_split(
    image_paths, image_labels, test_size=0.20, random_state=42, stratify=image_labels
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    val_test_paths, val_test_labels, test_size=0.50, random_state=42, stratify=val_test_labels
)

print(f"📊 Split Summary -> Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")

In [ ]:
# 4. High-Resolution Augmentation Pipeline (384x384) with Native Torchvision
import torchvision.transforms as transforms

IMAGE_SIZE = 384
BATCH_SIZE = 32  # Optimized for A100 / T4 GPU
NUM_WORKERS = 2

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class PlanticaDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        label = self.labels[idx]
        try:
            image = Image.open(path).convert('RGB')
        except Exception:
            # Corrupt image fallback
            image = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE))
        
        if self.transform:
            image = self.transform(image)
        
        return image, torch.tensor(label, dtype=torch.long)

train_dataset = PlanticaDataset(train_paths, train_labels, transform=train_transforms)
val_dataset = PlanticaDataset(val_paths, val_labels, transform=val_transforms)
test_dataset = PlanticaDataset(test_paths, test_labels, transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print("✅ Data Loaders Ready!")


In [ ]:
# 5. Build SOTA ConvNeXt Architecture with TIMM
MODEL_BACKBONE = 'convnext_base.fb_in22k_ft_in1k_384'  # State-of-the-Art pretrained on ImageNet-22k

def get_plantica_model(model_name=MODEL_BACKBONE, num_classes=num_classes):
    print(f"🏗️ Initializing TIMM Model: {model_name}")
    model = timm.create_model(
        model_name,
        pretrained=True,
        num_classes=num_classes,
        drop_rate=0.3,
        drop_path_rate=0.2
    )
    return model

model = get_plantica_model().to(device)
print("✅ Model Loaded Successfully into VRAM!")

In [ ]:
# 6. Training & Validation Engine (with AMP Mixed Precision & Label Smoothing)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

def train_one_epoch(model, dataloader, optimizer, criterion, scaler):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc="Training", leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(images)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += images.size(0)
        pbar.set_postfix({'loss': f"{loss.item():.4f}", 'acc': f"{correct/total*100:.2f}%"})
        
    return running_loss / total, (correct / total) * 100.0

def evaluate(model, dataloader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Validating", leave=False):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss = criterion(outputs, labels)
                
            running_loss += loss.item() * images.size(0)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += images.size(0)
            
    return running_loss / total, (correct / total) * 100.0

In [ ]:
# 7. Phase 1: Warmup Classifier Head (3 Epochs)
print("🔥 [PHASE 1] Freezing Backbone & Training Classifier Head...")
for name, param in model.named_parameters():
    if 'head' not in name and 'fc' not in name and 'classifier' not in name:
        param.requires_grad = False
    else:
        param.requires_grad = True

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-2)

for epoch in range(1, 4):
    t_loss, t_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    v_loss, v_acc = evaluate(model, val_loader, criterion)
    print(f"Phase 1 - Epoch [{epoch}/3] | Train Loss: {t_loss:.4f}, Acc: {t_acc:.2f}% | Val Loss: {v_loss:.4f}, Val Acc: {v_acc:.2f}%")

In [ ]:
# 8. Phase 2: Full Fine-Tuning with Cosine Annealing (12 Epochs)
print("🚀 [PHASE 2] Unfreezing Full Backbone with Cosine Scheduler...")
for param in model.parameters():
    param.requires_grad = True

EPOCHS_PHASE2 = 12
optimizer = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_PHASE2, eta_min=1e-7)

best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS_PHASE2 + 1):
    t_loss, t_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
    v_loss, v_acc = evaluate(model, val_loader, criterion)
    scheduler.step()
    
    history['train_loss'].append(t_loss)
    history['train_acc'].append(t_acc)
    history['val_loss'].append(v_loss)
    history['val_acc'].append(v_acc)
    
    print(f"Epoch [{epoch:02d}/{EPOCHS_PHASE2}] | Train Loss: {t_loss:.4f} (Acc: {t_acc:.2f}%) | Val Loss: {v_loss:.4f} (Val Acc: {v_acc:.2f}%)")
    
    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save(model.state_dict(), 'best_convnext_plantica.pth')
        print(f"  🏆 New Best Model Saved! (Val Acc: {best_val_acc:.2f}%)")

print(f"\n🎉 Training Complete! Peak Validation Accuracy: {best_val_acc:.2f}%")

In [ ]:
# 9. Comprehensive Test Set Evaluation
model.load_state_dict(torch.load('best_convnext_plantica.pth'))
model.eval()

all_preds = []
all_targets = []
all_probs = []

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing Best Model"):
        images = images.to(device)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
        
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
        all_targets.extend(labels.numpy())

all_probs = np.array(all_probs)
test_top1 = accuracy_score(all_targets, all_preds) * 100.0
test_top5 = top_k_accuracy_score(all_targets, all_probs, k=min(5, num_classes)) * 100.0

print("="*50)
print(f"🎯 Final Test Results:")
print(f"   Top-1 Accuracy: {test_top1:.2f}%")
print(f"   Top-5 Accuracy: {test_top5:.2f}%")
print("="*50)

In [ ]:
# 10. Install onnxscript & Export to High-Speed Production ONNX Model
!pip install -q onnx onnxscript onnxruntime

import torch
import json
import os
import zipfile

print("⚡ Exporting to Production ONNX format...")
model.load_state_dict(torch.load('best_convnext_plantica.pth', map_location='cpu'))
model.eval()
model.to('cpu')

dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device='cpu')
onnx_filename = 'plantica_convnext_387.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_filename,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

# Generate classes metadata mapping
classes_metadata = {
    "num_classes": num_classes,
    "classes": classes,
    "image_size": IMAGE_SIZE,
    "model_architecture": MODEL_BACKBONE
}

with open('classes.json', 'w', encoding='utf-8') as f:
    json.dump(classes_metadata, f, indent=2, ensure_ascii=False)

# Package into a clean zip file ready to download
zip_filename = 'plantica_convnext_artifacts.zip'
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(onnx_filename)
    zf.write('classes.json')
    zf.write('best_convnext_plantica.pth')

print(f"\n🎁 All artifacts zipped into '{zip_filename}'!")
print(f"File size: {os.path.getsize(zip_filename) / (1024*1024):.2f} MB")
print("👉 Downloading model artifacts...")

from google.colab import files
files.download(zip_filename)
